# COVID-19 clustering and logistic regression
Postgraduate group project • 89.5% overall assessment. Revised executable workflow. Custom implementations replace unavailable packages.

In [2]:
import pandas as pd,numpy as np,itertools as it, math
from pathlib import Path
from collections import Counter
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score,adjusted_rand_score,roc_auc_score,accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,roc_curve
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import RFECV,SelectFromModel
from sklearn.base import clone

base=Path.cwd(); data=base/'data'; m=c=base/'outputs'; m.mkdir(exist_ok=True)

## Mixed distance k prototypes, multiple starts

In [4]:
D=pd.read_csv(data/'D2.csv'); cols=['gender','weight','worried','income','smoking','working','insurance']; num=['weight','worried']; cat=[x for x in cols if x not in num]; A=StandardScaler().fit_transform(D[num]); B=np.column_stack([pd.factorize(D[x])[0] for x in cat]);gamma=.5*A.std(axis=0).mean()
def fit(A,B,k):
 best=None
 for seed in range(10):
  rng=np.random.default_rng(42+seed); ix=rng.choice(len(A),k,replace=False); a=A[ix].copy();b=B[ix].copy();prev=None
  for _ in range(100):
   dist=((A[:,None]-a)**2).sum(2)+gamma*(B[:,None]!=b).sum(2);y=dist.argmin(1)
   if prev is not None and np.array_equal(y,prev):break
   prev=y.copy()
   for j in range(k):
    mask=y==j
    if mask.any():a[j]=A[mask].mean(0);b[j]=[np.bincount(v).argmax() for v in B[mask].T]
    else:ix=rng.integers(len(A));a[j]=A[ix];b[j]=B[ix]
  dist=((A[:,None]-a)**2).sum(2)+gamma*(B[:,None]!=b).sum(2);y=dist.argmin(1);cost=dist.min(1).sum()
  if best is None or cost<best[0]:best=(cost,y)
 return best
ix=np.random.default_rng(42).choice(len(A),1500,replace=False);dist=euclidean_distances(A[ix],squared=True)+gamma*(B[ix,None,:]!=B[None,ix,:]).sum(2);np.fill_diagonal(dist,0)
rows=[]; labels={}
for k in [2,4,6,8]:
 cost,y=fit(A,B,k);labels[k]=y;rows.append([k,cost,silhouette_score(dist,y[ix],metric='precomputed')])
ks=pd.DataFrame(rows,columns=['k','cost','silhouette']);ks.to_csv(c/'clusters.csv',index=False);k=int(ks.loc[ks.silhouette.idxmax(),'k']);y=labels[k];cost2,y2=fit(A,np.column_stack([B,pd.factorize(D.age)[0]]),k);ari=adjusted_rand_score(y,y2)
for name,z in [('base',y),('age',y2)]:
 p=D.assign(cluster=z).groupby('cluster').agg(n=('covid19_positive','size'),positive=('covid19_positive','mean'),weight=('weight','mean'),worried=('worried','mean'),income=('income',lambda x:x.mode()[0]),working=('working',lambda x:x.mode()[0]));p.to_csv(c/(name+'.csv'))
print('CLUSTER',ks,k,ari,flush=True)

CLUSTER    k          cost  silhouette
0  2  13047.329968    0.468244
1  4   9018.048621    0.352462
2  6   7671.729755    0.343321
3  8   6927.256338    0.355437 2 1.0


## Supervised common split and train-only preprocessing

In [6]:
X=D.drop(columns='covid19_positive');Y=D.covid19_positive;xt,xv,yt,yv=train_test_split(X,Y,test_size=.3,random_state=10,stratify=Y); nums=X.select_dtypes('number').columns.tolist();cats=X.select_dtypes(exclude='number').columns.tolist();pre=ColumnTransformer([('num',StandardScaler(),nums),('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats)]);cv=StratifiedKFold(5,shuffle=True,random_state=10)
models={'Dummy':(DummyClassifier(strategy='prior'),{}),'Logistic default':(LogisticRegression(max_iter=1000),{}),'Logistic tuned':(LogisticRegression(max_iter=1000),{'model__C':[10.**i for i in range(-6,4)]})}
res=[];fitted={};plt.figure(figsize=(7,5))
def evaluate(name,pipe,params):
 search=GridSearchCV(pipe,params,cv=cv,scoring='roc_auc',n_jobs=1);search.fit(xt,yt);fitted[name]=search;pred=search.predict(xv);prob=search.predict_proba(xv)[:,1];tn,fp,fn,tp=confusion_matrix(yv,pred).ravel();res.append([name,search.best_score_,accuracy_score(yv,pred),precision_score(yv,pred,zero_division=0),recall_score(yv,pred),f1_score(yv,pred),roc_auc_score(yv,prob),tn,fp,fn,tp,str(search.best_params_)]);fpr,tpr,_=roc_curve(yv,prob);plt.plot(fpr,tpr,label=name);print(res[-1],flush=True)
for name,(model,params) in models.items():evaluate(name,Pipeline([('pre',clone(pre)),('model',model)]),params)

['Dummy', np.float64(0.5), 0.5964306275187105, 0.0, 0.0, 0.0, 0.5, np.int64(1036), np.int64(0), np.int64(701), np.int64(0), '{}']
['Logistic default', np.float64(0.7410730745174059), 0.6747265400115141, 0.6107491856677525, 0.5349500713266762, 0.5703422053231939, 0.7187484509167819, np.int64(797), np.int64(239), np.int64(326), np.int64(375), '{}']
['Logistic tuned', np.float64(0.7411267176318785), 0.6724237190558434, 0.610738255033557, 0.5192582025677603, 0.5612952968388589, 0.7200345342285428, np.int64(804), np.int64(232), np.int64(337), np.int64(364), "{'model__C': 0.1}"]


## Rfecv nested inside outer cv; feature selection performed only on training data

In [8]:
selector=RFECV(LogisticRegression(max_iter=1000),step=10,cv=3,scoring='roc_auc',min_features_to_select=5)
evaluate('Logistic RFECV',Pipeline([('pre',clone(pre)),('select',selector),('model',LogisticRegression(max_iter=1000))]),{})

plt.legend(fontsize=7);plt.xlabel('False positive rate');plt.ylabel('True positive rate');plt.tight_layout();plt.savefig(c/'roc.png',dpi=160);pd.DataFrame(res,columns=['model','cv_auc','accuracy','precision','recall','f1','auc','tn','fp','fn','tp','params']).to_csv(c/'metrics.csv',index=False)
print('DONE',flush=True)


['Logistic RFECV', np.float64(0.7408032036894842), 0.6747265400115141, 0.6107491856677525, 0.5349500713266762, 0.5703422053231939, 0.7187484509167819, np.int64(797), np.int64(239), np.int64(326), np.int64(375), '{}']
DONE


## Interpret results and inspect data

In [10]:
print(D.describe(include='all').T); print('Age comparison ARI:',ari); print(pd.read_csv(c/'base.csv')); print(pd.read_csv(c/'age.csv')); print(pd.read_csv(c/'metrics.csv')); ks.plot(x='k',y='silhouette',marker='o'); plt.savefig(c/'silhouette.png'); plt.show(); D.select_dtypes('number').hist(figsize=(10,7)); plt.tight_layout(); plt.savefig(c/'distributions.png'); plt.show(); import seaborn as sns; sns.heatmap(D.select_dtypes('number').corr(),annot=True,fmt='.2f'); plt.tight_layout(); plt.savefig(c/'correlation.png'); plt.show(); print('Selected features:',fitted['Logistic RFECV'].best_estimator_.named_steps['pre'].get_feature_names_out()[fitted['Logistic RFECV'].best_estimator_.named_steps['select'].support_])

                   count unique      top  freq  ...    25%    50%    75%    max
gender              5789      3     male  2985  ...    NaN    NaN    NaN    NaN
age                 5789      9    30_40  1412  ...    NaN    NaN    NaN    NaN
height            5789.0    NaN      NaN   NaN  ...  164.0  172.0  180.0  238.0
weight            5789.0    NaN      NaN   NaN  ...   68.0   82.0   96.0  180.0
blood_type          5789      9  unknown  1884  ...    NaN    NaN    NaN    NaN
insurance           5789      2      yes  5167  ...    NaN    NaN    NaN    NaN
income              5789      4      med  3417  ...    NaN    NaN    NaN    NaN
smoking             5789      8    never  3749  ...    NaN    NaN    NaN    NaN
alcohol           5789.0    NaN      NaN   NaN  ...    1.0    2.0    3.0   14.0
contacts_count    5789.0    NaN      NaN   NaN  ...    3.0    5.0   10.0   21.0
working             5789      5  stopped  1774  ...    NaN    NaN    NaN    NaN
worried           5789.0    NaN      NaN